In [55]:
import zipfile
import cv2
import numpy as np
from sklearn.preprocessing import StandardScaler

In [60]:
# Function to load images and labels from a zip file
def load_images_from_zip(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # Get the list of all files in the zip file
        image_files = [f for f in zip_ref.namelist() if not f.endswith('/') and '/.' not in f]  # Filter valid files
        X, y = [], []
        
        # Loop through the files and extract images and labels
        for file_name in image_files:
            with zip_ref.open(file_name) as file:
                img_data = file.read()
                img = cv2.imdecode(np.frombuffer(img_data, np.uint8), cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (64, 64))  # Resize images
                X.append(img)
                
                # Extract label from the folder structure (e.g., 'train/0/image_name.png' or 'train/1/image_name.png')
                split_path = file_name.split('/')
                if len(split_path) >= 3:  # Ensure valid paths
                    label = split_path[1]  # Folder name (0 or 1) is the label
                    y.append(int(label))  # Convert label to integer (0 or 1)
        
        # Convert X to a numpy array and normalize pixel values
        X = np.array(X) / 255.0  # Normalize pixel values to [0, 1]
        return X, np.array(y)

# Load images and labels directly from the zip file
zip_path = r'D:\Medical imaging diagnostics(breast cancer)\archive (9).zip'   # Update this with your zip path
X, y = load_images_from_zip(zip_path)

# Reshape images to flatten them for training a classifier
X = X.reshape(X.shape[0], -1)  # Flatten images into 1D vectors


In [61]:
# Scaling X and y
scaler_X = StandardScaler()
scaler_y = StandardScaler()

In [62]:
# Store the original mean and std of y before scaling
y_mean = np.mean(y)
y_std = np.std(y)

In [63]:
# Fit the scaler on X and y
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))  # Reshape y to 2D for scaling


In [66]:
def train_linear_regression(X, y, learning_rate, epochs):
    m = len(X)
    weights = np.zeros(X.shape[1])
    bias = 0

    for epoch in range(epochs):
        # Predictions
        y_pred = np.dot(X, weights) + bias
        
        # Compute error
        error = y_pred - y

        # Compute gradients
        dW = (1/m) * np.dot(X.T, error)
        dB = (1/m) * np.sum(error)

        # Print gradients every 100 epochs to check their magnitude
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, dW: {dW[:5]}, dB: {dB}")

        # Update weights and bias
        weights -= learning_rate * dW
        bias -= learning_rate * dB

        # Compute loss
        loss = (1 / (2 * m)) * np.sum(error ** 2)
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss}")
        
    return weights, bias


In [67]:
# Flatten y_scaled before passing it into the function
weights, bias = train_linear_regression(X_scaled, y_scaled.flatten(), learning_rate, epochs)


Epoch 0, dW: [0.02099674 0.03435435 0.0410769  0.03836186 0.04259993], dB: -1.34421327486392e-16
Epoch 0, Loss: 0.5
Epoch 100, dW: [-0.01560993 -0.00585382  0.0004734  -0.00226964  0.00155649], dB: 2.68842654972784e-16
Epoch 100, Loss: 0.49327593211694626
Epoch 200, dW: [-1.33300846e-02 -3.86194587e-03  2.61037302e-03  1.65273419e-05
  3.88476256e-03], dB: 3.02447986844382e-16
Epoch 200, Loss: 0.4905110865162586
Epoch 300, dW: [-0.01199702 -0.00279847  0.00376355  0.00126912  0.00515683], dB: 4.03263982459176e-16
Epoch 300, Loss: 0.4886007149017899
Epoch 400, dW: [-0.01118899 -0.00223154  0.00438241  0.00195717  0.00585312], dB: 4.36869314330774e-16
Epoch 400, Loss: 0.48712149561993207
Epoch 500, dW: [-0.010675   -0.00192971  0.00471042  0.00233536  0.0062339 ], dB: 4.70474646202372e-16
Epoch 500, Loss: 0.4858835746410222
Epoch 600, dW: [-0.01032722 -0.00176823  0.0048806   0.00254332  0.00644154], dB: 5.37685309945568e-16
Epoch 600, Loss: 0.48479339105630986
Epoch 700, dW: [-0.0100742

In [68]:
# Display some predictions
y_pred_scaled = np.dot(X_scaled, weights) + bias
print("\nSample Predictions (scaled):")
for i in range(5):
    print(f"Actual: {y[i]:.4f}, Predicted (scaled): {y_pred_scaled[i]:.4f}")



Sample Predictions (scaled):
Actual: 0.0000, Predicted (scaled): -0.0812
Actual: 0.0000, Predicted (scaled): 0.0961
Actual: 0.0000, Predicted (scaled): 0.1816
Actual: 0.0000, Predicted (scaled): -0.1317
Actual: 0.0000, Predicted (scaled): -0.1072


In [69]:
# Inverse scaling to get predictions in original scale
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))


In [71]:
print("\nSample Predictions (inverse scaled):")
for i in range(5):
    print(f"Actual: {y[i]:.4f}, Predicted: {y_pred[i][0]:.4f}")


Sample Predictions (inverse scaled):
Actual: 0.0000, Predicted: 0.3038
Actual: 0.0000, Predicted: 0.3879
Actual: 0.0000, Predicted: 0.4285
Actual: 0.0000, Predicted: 0.2798
Actual: 0.0000, Predicted: 0.2915


In [73]:
import cv2
import numpy as np
from sklearn.preprocessing import StandardScaler

# Function to predict a new image
def predict_new_image(image_path, model_weights, model_bias, scaler_X, scaler_y):
    # Load and preprocess the image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # Read the image in grayscale
    img = cv2.resize(img, (64, 64))  # Resize the image to the same dimensions as training data
    img = img / 255.0  # Normalize pixel values to [0, 1]
    
    # Flatten the image to a 1D array (same shape as training data)
    img_flat = img.reshape(1, -1)
    
    # Scale the image using the same scaler used during training
    img_scaled = scaler_X.transform(img_flat)
    
    # Predict using the trained model
    y_pred_scaled = np.dot(img_scaled, model_weights) + model_bias
    
    # Inverse scale the prediction to get the original value
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))

    # Display the numerical prediction and the corresponding label (if binary classification, for example)
    print(f"Predicted Number: {y_pred[0][0]:.4f}")
    
    if y_pred[0][0] < 0.5:
        label = "Class 0, The image suggests no signs of breast cancer."
    else:
        label = "Class 1, The image indicates potential signs of breast cancer. Please consult a doctor for further analysis."  
    
    print(f"Predicted Label: {label}")


image_path =  r"D:\Medical imaging diagnostics(breast cancer)\33_995020214_png.rf.4612f989168551a8c7c4145bf28bde1f.jpg"   # Replace with your image file path
model_weights = weights  # Use the trained weights from your model
model_bias = bias  # Use the trained bias from your model
scaler_X = scaler_X  # Scaler used for feature scaling (X data)
scaler_y = scaler_y  # Scaler used for target scaling (y data)

# Make a prediction for the new image
predict_new_image(image_path, model_weights, model_bias, scaler_X, scaler_y)


Predicted Number: 0.3940
Predicted Label: Class 0, The image suggests no signs of breast cancer.
